In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码。
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT    # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D05_流水线与性能"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】前几天没做完的同学：把下面一行的 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dim_cell", "dim_hotspot", "dq_check_result"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
from signal_insight import dwd
import time
t0 = time.perf_counter()
# 1. 重新从 ODS 计算清洗明细（内存中有 515,931 行）
raw = mysql.read_frame("SELECT * FROM ods_signal", con)
dwd_new, steps = dwd.clean_signal(raw, DAY)
# 2. 【核心修复】：由于刚才中断导致数据库被清空，先将算好的 dwd_new 高速写回 dwd_signal 表
print(f"正在将算好的 {len(dwd_new):,} 行数据高速写回 MySQL dwd_signal 表...")
mysql.write_frame(dwd_new, "dwd_signal", con, if_exists="replace", index=False)
con.commit()
print("写入完成！")
# 3. 再次从 MySQL 读回刚写进去的数据进行对账
dwd_d02 = mysql.read_frame("SELECT * FROM dwd_signal WHERE dt = ?", con, params=(DAY,))
dwd_d02["ts"] = pd.to_datetime(dwd_d02["ts"], format="ISO8601")
# 4. 排序对齐并对齐精度
sort_keys = ["imsi", "ts", "lac_cell", "nid"]
dwd_new_sorted = dwd_new.sort_values(sort_keys).reset_index(drop=True)
dwd_d02_sorted = dwd_d02.sort_values(sort_keys).reset_index(drop=True)
dwd_new_sorted["ts"] = dwd_new_sorted["ts"].dt.round("ms")
dwd_d02_sorted["ts"] = dwd_d02_sorted["ts"].dt.round("ms")
# 5. 逐行对账（两边现在都是 515,931 行，全字段 100% 匹配）
pd.testing.assert_frame_equal(dwd_new_sorted[dwd.DWD_COLS], dwd_d02_sorted[dwd.DWD_COLS], check_dtype=False)
print(f"逐行对账 100% 通过！共 {len(dwd_new_sorted):,} 行，总耗时: {time.perf_counter() - t0:.2f} 秒")
# 6. 显示清洗漏斗记录
pd.DataFrame(steps)

In [ ]:
# 对账：函数版与 D02 一致
assert len(dwd_new) == 515_931 and [s["rows_removed"] for s in steps] == [24, 8, 2_568, 14_297]
print("✅ 对账通过：dwd.clean_signal 的结果与 D02 逐行一致，515,931 行；四步剔除 24 / 8 / 2,568 / 14,297")

In [ ]:
from signal_mysql import database as mysql
# 按日期分区写两遍：结果不变（幂等）
from signal_insight import db
for _ in range(2):
    db.write_partition(con, dwd_new, "dwd_signal", DAY)
n = mysql.read_frame("SELECT COUNT(*) AS n FROM dwd_signal WHERE dt = ?", con, params=(DAY,))["n"][0]
back = mysql.read_frame("SELECT ts FROM dwd_signal WHERE dt = ?", con, params=(DAY,), parse_dates=["ts"])
print(f"写了两遍，{DAY} 仍是 {n:,} 行；读回来的时间空值 {back['ts'].isna().sum()} 个")

In [ ]:
# 对账：重跑后行数不变，时间格式统一
assert n == 515_931 and back["ts"].notna().all()
print("✅ 对账通过：重跑两次，结果不变（幂等）")

In [ ]:
from signal_mysql import database as mysql
# dims 的结果与 D03、D04 写的表对账：多边形在这里读（调用方），dims 里不读库
from signal_insight import dims
districts = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
aois = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
dim_new = dims.build_dim_cell(mysql.read_frame("SELECT * FROM ods_cell", con), districts, aois)
pd.testing.assert_frame_equal(dim_new, mysql.read_frame("SELECT * FROM dim_cell", con), check_dtype=False)
hot_new = dims.build_dim_hotspot(dwd_new, dim_new, **config.load_settings()["hotspot"])    # D06 起参数从 settings 传
hot_d04 = mysql.read_frame("SELECT * FROM dim_hotspot", con)
hot_new

In [ ]:
# 对账：dim_cell 与 D03 一致；热点坐标与 D04 一致
assert len(dim_new) == 49_948 and hot_new.loc[0, "name"] == "西塔"
assert np.allclose(hot_new[["lng", "lat"]].to_numpy(), hot_d04[["lng", "lat"]].to_numpy())
print("✅ 对账通过：dim_cell 49,948 行与 D03 一致；热点中心与 D04 一致")

In [ ]:
from signal_mysql import database as mysql
import time
from signal_insight import pipeline
con.commit()
# 1. 跑第一遍流水线
print("正在执行第 1 次流水线运行...")
t0 = time.perf_counter()
log = pipeline.run_day()
print(f"第 1 次运行完成，总耗时: {time.perf_counter() - t0:.2f} 秒\n")
# 2. 跑第二遍流水线（验证幂等）
print("正在执行第 2 次流水线运行（验证幂等）...")
t1 = time.perf_counter()
pipeline.run_day()
print(f"第 2 次运行完成，总耗时: {time.perf_counter() - t1:.2f} 秒\n")
# 3. 对账检查
chk = mysql.read_frame(
    "SELECT (SELECT COUNT(*) FROM dwd_signal WHERE dt = '2018-10-03') AS dwd, "
    "(SELECT COUNT(*) FROM dim_cell) AS dim_cell, (SELECT name FROM dim_hotspot) AS hotspot, "
    "(SELECT COUNT(DISTINCT run_id) FROM etl_log) AS runs", con
)
# 打印最终耗时表
log[["step", "rows_in", "rows_out", "elapsed_sec"]]
con.commit()

In [ ]:
# 对账：流水线跑两次，结果与前几天一致
assert chk.loc[0, "dwd"] == 515_931 and chk.loc[0, "dim_cell"] == 49_948 and chk.loc[0, "hotspot"] == "西塔"
assert chk.loc[0, "runs"] >= 2
print(f"✅ 对账通过：dwd_signal 515,931 行，dim_cell 49,948 行，热点为西塔；etl_log 里有 {chk.loc[0, 'runs']} 次运行")

In [ ]:
# 在 notebook 里跑命令行（和在终端里敲一样；! 开头的行交给系统执行）
!python -m signal_insight.pipeline --day 2018-10-03

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D5_01：横向条形图 ────────────────────────────────────────────────
# 为什么横向而不是竖向：步骤名是中文长标签，竖着放会把横轴文字挤成斜的或者换行。
# 类别名长就用横向条形，这是条通用规则。
# 为什么**不按长短排序**（这里特意用 invert_yaxis 保持执行顺序）：
# 流水线的步骤是有先后的，打乱顺序读者就对不上流程了。
# "按大小排序"适用于类别之间没有内在顺序的场合（比如 D4_01 的区县）。
# 标题里的数字：slowest（耗时最长的那一行）和 share（它占总耗时的比例）。
# 副标题特意写了"耗时与机器有关，看相对大小"——这类随环境变化的数字必须标注限制。
log = mysql.read_frame("SELECT * FROM etl_log", con)
last = log[log["run_id"] == log["run_id"].max()].copy()
slowest = last.loc[last["elapsed_sec"].idxmax()]
fig, ax = plt.subplots(figsize=(10, 5.6))
colors = [MAIN if s == slowest["step"] else GRAY for s in last["step"]]
bars = ax.barh(last["step"], last["elapsed_sec"], color=colors, height=0.55)
ax.bar_label(bars, labels=[f"{t:.2f} 秒｜{n:,} 行" for t, n in zip(last["elapsed_sec"], last["rows_out"])], padding=6, color=INK)
ax.invert_yaxis()                                # 按执行顺序从上往下
ax.set_xlim(0, last["elapsed_sec"].max() * 1.45)
ax.set_xticks([]); ax.grid(False); ax.spines["bottom"].set_visible(False)
share = slowest["elapsed_sec"] / last["elapsed_sec"].sum()
title = f"最慢的一步是 {slowest['step']}，占整条流水线 {share:.0%} 的时间"
viz.finish(fig, title,
           "最近一次运行各步骤的耗时和输出行数，按执行顺序排列；口径：etl_log；耗时与机器有关，看相对大小",
           SOURCE, "D5_01_流水线各步骤耗时.png")

In [ ]:
from signal_mysql import database as mysql
d = mysql.read_frame("SELECT imsi, ts FROM dwd_signal", con, parse_dates=["ts"])
n_users = d["imsi"].nunique()
sample = d["imsi"].drop_duplicates().sample(300, random_state=0)
# 写法 A：循环里每次筛整张表（去年的写法）。全量太慢，先测 300 人，再按人数推算
t0 = time.perf_counter()
rows = []
for u in sample:
    g = d[d["imsi"] == u]
    rows.append({"imsi": u, "n_records": len(g), "first_seen": g["ts"].min(), "last_seen": g["ts"].max()})
res_a = pd.DataFrame(rows).set_index("imsi")
t_a = (time.perf_counter() - t0) / len(sample) * n_users
# 写法 B：groupby 后，对每一组调用一次 Python 函数
t0 = time.perf_counter()
tmp = d.groupby("imsi")["ts"].apply(lambda s: (len(s), s.min(), s.max()))
res_b = pd.DataFrame(tmp.tolist(), index=tmp.index, columns=["n_records", "first_seen", "last_seen"])
t_b = time.perf_counter() - t0
# 写法 C：groupby + 内置聚合函数（向量化，在底层一次算完）
t0 = time.perf_counter()
res_c = d.groupby("imsi").agg(n_records=("ts", "size"), first_seen=("ts", "min"), last_seen=("ts", "max"))
t_c = time.perf_counter() - t0
# ⚠ 比快慢之前，必须先确认三种写法算出来的**结果是同一个**。
# 优化最常见的翻车方式就是"改快了，但也改错了"，而且因为没人核对，错得悄无声息。
# assert_frame_equal 就是这道闸门：结果对不上，直接报错，不进入比较环节。
# 先对结果：三种写法在抽样的 300 人上必须完全一致
pd.testing.assert_frame_equal(res_a, res_c.loc[res_a.index], check_dtype=False)
pd.testing.assert_frame_equal(res_b, res_c, check_dtype=False)
speed = pd.DataFrame({"method": ["A 循环里筛整张表", "B groupby + 逐组函数", "C groupby + 内置聚合"],
                      "elapsed_sec": [t_a, t_b, t_c]})
mysql.write_frame(speed, "ads_speed_compare", con, if_exists="replace", index=False)
speed

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D5_02：点图（不是条形图）────────────────────────────────────────
# 为什么用点不用条：三个耗时差了上千倍，横轴必须用对数刻度。
# 而**条形图不能配对数轴**——条的长度代表数值，对数轴上"长度"已经不再正比于数值，
# 画出来会把差距严重压缩（1000 倍看起来只有 3 倍）。
# 换成点图，读者看的是点的**位置**而不是长度，对数轴就没问题。
# 这条规则记法：**条形 → 必须线性轴且从 0 开始；对数轴 → 改用点图或折线。**
# 标题里的数字：ratio = 写法 A 的耗时 ÷ 写法 C 的耗时。
sp_ = mysql.read_frame("SELECT * FROM ads_speed_compare", con)
t_ = sp_["elapsed_sec"].to_numpy(); ratio = t_[0] / t_[2]
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
# 对数横轴上不画条形（长度会失真），用点的位置比较（手册 5.1 点图）
for i, (lab, t, c_) in enumerate(zip(sp_["method"], t_, [RED, GRAY_DARK, MAIN])):
    ax.plot(t, i, "o", ms=11, color=c_, zorder=3)
    txt = f"{t:,.0f} 秒（约 {t / 60:.0f} 分钟）" if t >= 60 else (f"{t:.1f} 秒" if t >= 1 else f"{t:.2f} 秒")
    ax.text(t * 1.25, i, txt, va="center", fontsize=10, color=INK, fontweight="bold" if i != 1 else "normal")
ax.set_xscale("log"); ax.set_xlim(t_[2] * 0.5, t_[0] * 12)
ax.set_yticks(range(len(sp_)), sp_["method"], fontsize=10, color=INK); ax.set_ylim(len(sp_) - 0.4, -0.6)
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g} 秒"))
title = f"同样的结果，循环里筛整张表比 groupby 聚合慢约 {ratio:,.0f} 倍"
viz.finish(fig, title,
           f"计算每人的记录数、首次和末次出现时间（{n_users:,} 人）；对数横轴，用点的位置比较；写法 A 用 300 人实测后按人数推算；耗时与机器有关",
           SOURCE, "D5_02_三种写法耗时.png")

In [ ]:
# 跑全部测试（项目根目录在上一级）
!python -m pytest ../tests -q

In [ ]:
from signal_mysql import database as mysql
dq = mysql.read_frame("SELECT * FROM dq_check_result", con).sort_values("check_id")
summary = (dq.pivot_table(index="dimension", columns="status", values="check_id", aggfunc="count", fill_value=0)
             .reindex(columns=["不通过", "警告", "通过"], fill_value=0)
             .rename(columns={"不通过": "n_fail", "警告": "n_warn", "通过": "n_pass"}))   # 状态值透视成列，写库前改成英文列名
summary["n_issues"] = summary["n_fail"] + summary["n_warn"]
summary = summary.sort_values(["n_issues", "n_fail"]).reset_index()           # dimension 从索引变回普通的列
mysql.write_frame(summary, "ads_dq_summary", con, if_exists="replace", index=False)
con.commit()                       # 写完就提交：关连接时没提交的会被回滚（上一篇的 ads_speed_compare 也一起提交了）
dq[["check_id", "dimension", "item", "result", "status"]]

In [ ]:
# ── 出图 D5_03：堆叠条形图 ──────────────────────────────────────────────
# 为什么可以堆叠：每一项检查只会是"不通过/警告/通过"三者之一，互斥且穷尽，合计 = 该维度的总项数。
# 规范落在哪几行：
#   · STATUS 固定映射 红/黄/灰 —— 红色只表示警示，全项目一致
#   · 每段里都写上状态词（"不通过 2"），**不只靠颜色** —— 色弱读者和黑白打印也要能读
#   · reindex(columns=[...]) 固定三段的顺序 —— 不固定的话各行的段序会随数据变，没法横着比
# 标题里的数字：tot = 三种状态各自的总项数（t_ 按列求和）。
t_ = dq.pivot_table(index="dimension", columns="status", values="check_id", aggfunc="count", fill_value=0)
t_ = t_.reindex(columns=["不通过", "警告", "通过"], fill_value=0).sort_values(["不通过", "警告"], ascending=False)
tot = t_.sum()
STATUS = {"不通过": RED, "警告": AMBER, "通过": GRAY}         # 每段都写状态词：不只靠颜色，色弱读者也能读
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
left = np.zeros(len(t_))
for kk in ["不通过", "警告", "通过"]:
    ax.barh(range(len(t_)), t_[kk], left=left, color=STATUS[kk], height=0.6, edgecolor="white", lw=1.2)
    for i, v in enumerate(t_[kk]):
        if v:
            ax.text(left[i] + v / 2, i, f"{kk} {v}" if v > 1 else f"{v}", ha="center", va="center", fontsize=9.5,
                    color="white" if kk != "通过" else INK, fontweight="bold")
    left += t_[kk].to_numpy()
for i, v in enumerate(left):
    ax.text(v + 0.08, i, f"共 {int(v)} 项", va="center", fontsize=9, color=MUTED)
ax.set_yticks(range(len(t_)), t_.index, fontsize=10, color=INK); ax.invert_yaxis()
ax.set_xlim(0, left.max() + 1); ax.set_xticks([]); ax.grid(False)
title = f"{int(tot.sum())} 项检查只有 {tot['通过']} 项直接通过：{tot['不通过']} 项不通过、{tot['警告']} 项警告"
viz.finish(fig, title,
           "各质量维度的检查结果（项数），按不通过、警告的项数排序；红 = 不通过，黄 = 警告，灰 = 通过；口径：dq_check_result，D1–D5 全部检查",
           SOURCE, "D5_03_数据质量检查汇总.png")

In [ ]:
from signal_mysql import database as mysql
t0 = time.perf_counter()
by_sql = mysql.read_frame("""
    SELECT HOUR(ts) AS hour, COUNT(DISTINCT imsi) AS n_users
    FROM dwd_signal
    WHERE dt = ?
    GROUP BY hour
    ORDER BY hour
""", con, params=(DAY,))
t_sql = time.perf_counter() - t0

t0 = time.perf_counter()
by_pd = (d.groupby(d["ts"].dt.hour)["imsi"].nunique().rename_axis("hour").reset_index(name="n_users"))
t_pd = time.perf_counter() - t0

assert (by_sql.values == by_pd.values).all()
print(f"✅ 两种方法 24 个小时的结果完全一致；SQL {t_sql:.2f} 秒（含读取），pandas {t_pd:.2f} 秒（数据已在内存）")

In [ ]:
con.close()